In [0]:
# Import required PySpark functions
from pyspark.sql import functions as F
from pyspark.sql.functions import expr, col, coalesce, lit, monotonically_increasing_id, when

In [0]:
# Load raw for-hire vehicle data
df_init = spark.read.table("group3_gp.bronze.for_hire_vehicles")

In [0]:
%sql
INSERT INTO group3_gp.gold.tabel_count
SELECT 
    'FHV' AS source,
    COUNT(*) AS row_count,
    'Ingest' AS reason
FROM group3_gp.bronze.for_hire_vehicles;

In [0]:
# Drop columns not needed for the silver layer
cols_to_drop = [
    "sr_flag",
    "affiliated_base_number",
    "dispatching_base_num"
]

df_clean = df_init.drop(*cols_to_drop)

In [0]:
# Cast timestamps, add service/taxi type labels, and convert location IDs to integer
column_types = {
    "pickup_datetime": "timestamp",
    "dropoff_datetime": "timestamp"
}

df = df_clean

for c, dtype in column_types.items():
    if c in df.columns:
        df = df.withColumn(c, expr(f"try_cast({c} as {dtype})"))

df = (
    df
    .withColumn("service_type", lit("Unknown"))
    .withColumn("taxi_type", lit("Small_volume_fhv"))
    .withColumn("pulocationid", col("pulocationid").cast("double").cast("int"))
    .withColumn("dolocationid", col("dolocationid").cast("double").cast("int"))
)

In [0]:
# Filter out invalid rows: missing timestamps, bad time ranges, null locations
final_df = df.filter(
    F.col("pickup_datetime").isNotNull() &
    F.col("dropoff_datetime").isNotNull() &
    (F.col("dropoff_datetime") > F.col("pickup_datetime")) &
    F.col("pulocationid").isNotNull() |
    F.col("dolocationid").isNotNull()
)

In [0]:
final_df.createOrReplaceTempView("final_df")

In [0]:
%sql
INSERT INTO group3_gp.gold.tabel_count
SELECT 
    'FHV_bronze' AS source,
    COUNT(*) AS row_count,
    'Drop rows missing crucial data, such as datetime, geodata, and impossible values' AS reason
FROM final_df;

In [0]:
# Create silver table from cleaned and enriched data
final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.silver.fhv_taxi_trips")